## Custom MCP Server in LangChain Host

In [ ]:
%pip install -q langchain langchain-openai langchain-mcp-adapters python-dotenv pywin32

In [ ]:
%pip install --force-reinstall --no-cache-dir pywin32 

# restart the kernel after running the above command to ensure that the 
# pywin32 module is properly registered and available for use in your Python environment.

In [1]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient  
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, ToolMessage

from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)  # Load environment variables from .env file

True

In [ ]:
# EXPLICITLY RUN THE WEATHER MCP SERVER LOCALLY using python mcp_weather_server.py COMMAND
# OR RUN THE BELOW CELL

In [ ]:
# import subprocess, sys, time, socket

# # Jupyter can't spawn stdio servers (its stderr has no fileno), so run the server as an HTTP process
# server_proc = subprocess.Popen(
#     [sys.executable, "mcp_weather_server.py"],
#     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
# )
# for _ in range(50):
#     try:
#         socket.create_connection(("127.0.0.1", 8000), timeout=0.5).close()
#         break
#     except OSError:
#         time.sleep(0.2)
# print("Weather MCP server running:", server_proc.poll() is None)

Weather MCP server running: True


In [3]:
model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)


async def get_mcp_client():

    return MultiServerMCPClient(
        {
            "weather": {
                "transport": "streamable_http",  # HTTP server started in the previous cell
                "url": "http://127.0.0.1:8000/mcp"
            }
        }
    )

async def get_agent():
    client = await get_mcp_client()
    tools = await client.get_tools()
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt="""You are a helpful assistant. 
        You can use the Weather tool to get the weather information."""
    )
    return agent


In [4]:
agent = await get_agent()

In [5]:
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in Seattle?"}]}
)
print(f"Agent response : {response['messages'][-1].content}")


Agent response : It’s always sunny in Seattle.
